In [1]:
%pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

In [3]:
DATA_PATH = Path("../data/data_final.csv")

big_df = pd.read_csv(DATA_PATH)
big_df["text"] = big_df["text"].fillna("").astype(str)
big_df["category"] = big_df["category"].fillna("").astype(str)

print("Rows loaded:", len(big_df))
print("Categories before dropping Undefined:")
print(big_df["category"].value_counts())

big_df = big_df[big_df["category"] != "Undefined"].reset_index(drop=True)

print("\nRows after dropping Undefined:", len(big_df))
counts = big_df["category"].value_counts()
print(counts)
print(f"Imbalance ratio (max/min): {counts.max() / counts.min():.1f}x")

imbalance_ratio = counts.max() / counts.min()
use_class_weights = imbalance_ratio > 10
print(f"Use class weights: {use_class_weights}")

C:\Users\oofuc\AppData\Local\Temp\ipykernel_9000\3080509077.py:3: DtypeWarning: Columns (0: category_8_raw) have mixed types. Specify dtype option on import or set low_memory=False.
  big_df = pd.read_csv(DATA_PATH)


Rows loaded: 365226
Categories before dropping Undefined:
category
Entertainment    80426
Undefined        44042
Business         42152
Politics         41587
World            40440
Sports           39025
Technology       38851
Health           27978
Science          10725
Name: count, dtype: int64

Rows after dropping Undefined: 321184
category
Entertainment    80426
Business         42152
Politics         41587
World            40440
Sports           39025
Technology       38851
Health           27978
Science          10725
Name: count, dtype: int64
Imbalance ratio (max/min): 7.5x
Use class weights: False


In [4]:
train_df, temp_df = train_test_split(
    big_df,
    test_size=0.3,
    stratify=big_df["category"],
    random_state=42,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["category"],
    random_state=42,
)

train_df = train_df[["text", "category", "source_dataset"]].reset_index(drop=True)
val_df = val_df[["text", "category", "source_dataset"]].reset_index(drop=True)
test_df = test_df[["text", "category", "source_dataset"]].reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain category distribution:")
print(train_df["category"].value_counts())
print("\nValidation category distribution:")
print(val_df["category"].value_counts())
print("\nTest category distribution:")
print(test_df["category"].value_counts())

Train shape: (224828, 3)
Validation shape: (48178, 3)
Test shape: (48178, 3)

Train category distribution:
category
Entertainment    56298
Business         29506
Politics         29111
World            28308
Sports           27317
Technology       27196
Health           19585
Science           7507
Name: count, dtype: int64

Validation category distribution:
category
Entertainment    12064
Business          6323
Politics          6238
World             6066
Sports            5854
Technology        5828
Health            4196
Science           1609
Name: count, dtype: int64

Test category distribution:
category
Entertainment    12064
Business          6323
Politics          6238
World             6066
Sports            5854
Technology        5827
Health            4197
Science           1609
Name: count, dtype: int64


In [5]:
le = LabelEncoder()
y_train = le.fit_transform(train_df["category"])
y_val = le.transform(val_df["category"])
y_test = le.transform(test_df["category"])

print("Label order:", list(le.classes_))
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

class_indices = np.arange(len(le.classes_))
class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=class_indices,
    y=y_train,
)
class_weights = dict(zip(class_indices, class_weights_values))
print("Class weights:", class_weights)

Label order: ['Business', 'Entertainment', 'Health', 'Politics', 'Science', 'Sports', 'Technology', 'World']
y_train shape: (224828,)
y_val shape: (48178,)
y_test shape: (48178,)
Class weights: {np.int64(0): np.float64(0.9524672947875008), np.int64(1): np.float64(0.49919180077445025), np.int64(2): np.float64(1.4349502170028083), np.int64(3): np.float64(0.9653910892789667), np.int64(4): np.float64(3.743639270014653), np.int64(5): np.float64(1.0287915949774866), np.int64(6): np.float64(1.0333688777761436), np.int64(7): np.float64(0.9927758937402854)}


In [6]:
SPLITS_DIR = Path("../data/splits")
ARTIFACTS_DIR = Path("../data/artifacts")
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(SPLITS_DIR / "train.csv", index=False, encoding="utf-8")
val_df.to_csv(SPLITS_DIR / "val.csv", index=False, encoding="utf-8")
test_df.to_csv(SPLITS_DIR / "test.csv", index=False, encoding="utf-8")

with open(ARTIFACTS_DIR / "label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

with open(ARTIFACTS_DIR / "class_weights.json", "w", encoding="utf-8") as f:
    json.dump({str(k): float(v) for k, v in class_weights.items()}, f, indent=2)

print("Wrote:")
print(" ", SPLITS_DIR / "train.csv")
print(" ", SPLITS_DIR / "val.csv")
print(" ", SPLITS_DIR / "test.csv")
print(" ", ARTIFACTS_DIR / "label_encoder.pkl")
print(" ", ARTIFACTS_DIR / "class_weights.json")

Wrote:
  ..\data\splits\train.csv
  ..\data\splits\val.csv
  ..\data\splits\test.csv
  ..\data\artifacts\label_encoder.pkl
  ..\data\artifacts\class_weights.json


In [7]:
print("Final prep summary:")
print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))
print("Number of classes:", len(le.classes_))
print("Classes:", list(le.classes_))
print("Class weights enabled:", use_class_weights)
print("Class weights:", class_weights)

Final prep summary:
Training samples: 224828
Validation samples: 48178
Test samples: 48178
Number of classes: 8
Classes: ['Business', 'Entertainment', 'Health', 'Politics', 'Science', 'Sports', 'Technology', 'World']
Class weights enabled: False
Class weights: {np.int64(0): np.float64(0.9524672947875008), np.int64(1): np.float64(0.49919180077445025), np.int64(2): np.float64(1.4349502170028083), np.int64(3): np.float64(0.9653910892789667), np.int64(4): np.float64(3.743639270014653), np.int64(5): np.float64(1.0287915949774866), np.int64(6): np.float64(1.0333688777761436), np.int64(7): np.float64(0.9927758937402854)}
